# Aula 3 — Este modelo presta?

**Disciplina 2 · Aprendizado de Máquina · IASEG**

Ontem a gente parou no meio da regressão: a base estava auditada, as colunas escolhidas,
e faltava o modelo. Hoje a gente termina, e passa para a etapa que dá nome à aula.

| | Etapa | |
|---|---|---|
| 1 | Definição do problema | |
| 2 | Preparação dos dados | |
| 3 | Divisão treino / teste | |
| 4 | Pré-processamento | |
| 5 | Treinamento do modelo | |
| 6 | Predição | |
| **7** | **Avaliação** | **← hoje** |

**A pergunta da aula:** como eu sei se um modelo é bom?

- **Parte 1** — Terminar a regressão da idade, e julgar o resultado.
- **Parte 2** — Acertou ou errou? Ou: por quanto errou?
- **Parte 3** — A pergunta do próprio banco: quem aceita o investimento?

**Onde aparecer `___`, é uma decisão de vocês.** Não é sintaxe: é escolher o que o modelo deve fazer.

## Configuração

`pandas` para os dados. O resto é importado na etapa em que for usado.

In [1]:
import pandas as pd

---

# Parte 1 — A regressão de ontem, até o fim

## Onde paramos ontem

> **Dá para prever a _idade_ de uma pessoa a partir do perfil dela?**

A saída é um **número** → **regressão**. Cada linha tem a idade certa → **supervisionado**.

A auditoria da base foi feita ontem. Paramos na decisão de **o que entra no modelo**.

## Etapa 2 — Preparação dos dados

Uma linha, uma pessoa para quem o banco ligou. O arquivo é separado por **ponto e vírgula**.

In [3]:
URL = "https://raw.githubusercontent.com/HegdeChaitra/Bank-Marketing-Campaign-Analysis/master/bank-additional-full.csv"

df = pd.read_csv(URL, sep=';')

print('Linhas:', df.shape[0], '| Colunas:', df.shape[1])

Linhas: 41188 | Colunas: 21


### O que entra no modelo

**As seis colunas que descrevem a pessoa:** `job`, `marital`, `education`, `default`,
`housing`, `loan`.

- As colunas de economia (`euribor3m`, `nr.employed`...) descrevem **o país**.
- As de campanha (`campaign`, `pdays`, `contact`...) descrevem **o que o banco fez**.

Podia ser diferente. Mas a escolha tem que estar escrita no relatório.

In [4]:
COLUNAS = ['job', 'marital', 'education', 'default', 'housing', 'loan']

X = df[COLUNAS]      # as entradas (features)
y = df['age']        # o alvo a prever: a idade

print('X:', X.shape, '| y:', y.shape)
X.head()

X: (41188, 6) | y: (41188,)


,job,marital,education,default,housing,loan
0,housemaid,married,basic.4y,no,no,no
1,services,married,high.school,unknown,no,no
2,services,married,high.school,no,yes,no
3,admin.,married,basic.6y,no,no,no
4,services,married,high.school,no,no,yes


### Texto vira número: uma coluna de 0 e 1 para cada valor

A máquina só trabalha com números. Como transformar o estado civil em número?

Se a gente escrevesse *divorciado = 0, casado = 1, solteiro = 2*, o modelo leria
"solteiro" como **o dobro** de "casado", o que não quer dizer nada.

Em vez disso, cada valor vira **uma coluna de sim ou não**: ou a pessoa é casada ou não é;
ou é solteira ou não é. Isso se chama ***one-hot encoding***.

In [5]:
X_dummies = pd.get_dummies(X, drop_first=True)

print('Antes:', X.shape[1], 'colunas de texto')
print('Depois:', X_dummies.shape[1], 'colunas de 0 e 1')
X_dummies.head()

Antes: 6 colunas de texto
Depois: 27 colunas de 0 e 1


,job_blue-collar,job_entrepreneur,job_housemaid,job_management,job_retired,job_self-employed,job_services,job_student,job_technician,job_unemployed,...,education_illiterate,education_professional.course,education_university.degree,education_unknown,default_unknown,default_yes,housing_unknown,housing_yes,loan_unknown,loan_yes
0,False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,True,False,False,False,...,False,False,False,False,True,False,False,False,False,False
2,False,False,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,True,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,True


Seis colunas de texto viraram **27** colunas de 0 e 1.

**Mas duas delas são a mesma coluna.** `housing_unknown` e `loan_unknown` marcam as mesmas
pessoas: quem não tem a informação do financiamento também não tem a do empréstimo.

In [6]:
iguais = (X_dummies['housing_unknown'] == X_dummies['loan_unknown']).all()

print('As duas colunas são idênticas?', iguais)
print('Pessoas marcadas:', X_dummies['housing_unknown'].sum())

As duas colunas são idênticas? True
Pessoas marcadas: 990


Uma coluna repetida não traz informação nova, e atrapalha a conta do modelo. Tiramos uma:
ficam **26** colunas. *(É o tipo de coisa que a auditoria procura.)*

In [7]:
X_cod = X_dummies.drop(columns='loan_unknown')

print('Colunas que entram no modelo:', X_cod.shape[1])

Colunas que entram no modelo: 26


> **Por que isto está na Etapa 2 e não na 4?** Aqui não se aprende nada com os dados:
> só se reescreve a tabela num formato que o modelo entende. Por isso pode ser feito antes
> da divisão treino/teste, sem risco de vazamento.

## Etapa 3 — Divisão treino / teste

Igual ao Iris: 80% para treinar, 20% para testar, `random_state=42` para a divisão ser
sempre a mesma. *(Sem `stratify`: ele mantém a proporção de categorias, e o alvo aqui é um número.)*

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_cod, y, test_size=0.2, random_state=42)

print('Treino:', X_train.shape[0], 'linhas | Teste:', X_test.shape[0], 'linhas')

Treino: 32950 linhas | Teste: 8238 linhas


## Etapa 4 — Pré-processamento

**Hoje não tem nada a fazer aqui.** No Iris a gente padronizou porque o k-NN mede distâncias;
a regressão linear não mede distância.

A etapa continua na lista mesmo sem fazer nada: o pipeline é uma **checagem**, não uma receita obrigatória.

## Etapa 5 — Treinamento

Uma linha para achar os parâmetros: os números que fazem a reta errar menos nos dados de treino.

In [9]:
from sklearn.linear_model import LinearRegression

modelo = LinearRegression()
modelo.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](26,)","[-1.92, 1.23, 2.46,...,-0.11, 0.14,-0.13]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](26,)","['job_blue-collar','job_entrepreneur','job_housemaid',..., 'housing_unknown','housing_yes','loan_yes']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,47.45
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,26
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(26)


## Etapa 6 — Predição

O modelo diz uma idade para cada pessoa do teste, e a gente coloca lado a lado com a idade real.

In [10]:
y_pred = modelo.predict(X_test)

comparacao = pd.DataFrame({
    'idade real': y_test.values,
    'idade prevista': y_pred.round(1)})

comparacao.head(10)

,idade real,idade prevista
0,57,39.3
1,55,50.5
2,33,38.4
3,36,39.7
4,27,42.3
5,58,59.7
6,48,41.2
7,51,45.1
8,24,41.2
9,36,42.7


Uma pessoa de 58 anos virou 59,7: quase. Uma de 24 virou 41,2: longe.
**Ele errou?** Guardem essa pergunta.

---

## O modelo é uma soma

O modelo inteiro são **27 números**: um **ponto de partida** e um **peso** para cada uma das
26 colunas. Como o alvo é idade, os pesos estão em **anos**.

In [11]:
pesos = pd.Series(modelo.coef_, index=X_cod.columns)

print(f'Ponto de partida: {modelo.intercept_:.1f} anos')
print('Quantidade de pesos:', len(pesos))

Ponto de partida: 47.5 anos
Quantidade de pesos: 26


### Uma previsão feita à mão

Uma pessoa **aposentada**, **casada**, com **ensino médio** (`high.school`), **sem**
inadimplência, **com** financiamento de casa, **sem** empréstimo pessoal.

> **Antes de rodar:** partindo de 47,5 anos e somando os pesos das colunas ligadas para essa
> pessoa, quanto o modelo vai dizer?
>
> **Palpite da turma: ______ anos**

In [12]:
pessoa = pd.DataFrame([{'job': 'retired', 'marital': 'married',
                        'education': 'high.school', 'default': 'no',
                        'housing': 'yes', 'loan': 'no'}])

# A mesma pessoa, escrita com as mesmas 26 colunas de 0 e 1
pessoa_cod = pd.get_dummies(pessoa).reindex(columns=X_cod.columns, fill_value=False)

# Os pesos das colunas que estão ligadas (= 1) para essa pessoa
ligados = pesos[pessoa_cod.iloc[0].astype(bool)]

print(f'{"ponto de partida":24s} {modelo.intercept_:6.1f}')
for coluna, peso in ligados.items():
    print(f'{coluna:24s} {peso:+6.1f}')
print(f'{"soma feita à mão":24s} {modelo.intercept_ + ligados.sum():6.1f}')
print()
print(f'O que o modelo diz:      {modelo.predict(pessoa_cod)[0]:6.1f}')

ponto de partida           47.5
job_retired               +19.0
marital_married            -2.3
education_high.school      -5.4
housing_yes                +0.1
soma feita à mão           58.9

O que o modelo diz:        58.9


As outras colunas estão desligadas (0 vezes o peso = 0), então não somam nada.

**É só isso que o modelo faz.** Esses 27 números **são** o modelo, e por isso dava para
apagar as 41.188 linhas depois do treino.

*Aposentado soma 19 anos. Faz sentido?* O que cada peso quer dizer, e o que dá para afirmar
a partir deles, fica para a **aula 7**.

### Pessoas com exatamente esse perfil

No teste existem **doze** pessoas com essas mesmas seis respostas. O que o modelo diz para elas?

In [13]:
mesmo_perfil = (X_test == pessoa_cod.iloc[0]).all(axis=1)

pd.DataFrame({
    'idade real': y_test[mesmo_perfil],
    'idade prevista': y_pred[mesmo_perfil.values].round(1)})

,idade real,idade prevista
6314,55,58.9
38202,63,58.9
36779,55,58.9
17284,59,58.9
39482,61,58.9
37805,58,58.9
19785,58,58.9
14803,49,58.9
21716,56,58.9
24347,58,58.9


**Mesmo perfil, mesma resposta: 58,9 para todas.** As idades reais vão de 49 a 77.
O erro não é falta de dados: essas seis colunas simplesmente não são o bastante para acertar a idade de ninguém.
Mesmo com todas as pessoas do mundo na base, o modelo continuaria errando essas pessoas.

---

## Etapa 7 — Avaliação: quanto ele erra?

O erro de uma pessoa é a **distância** entre a idade real e a prevista, sem importar se o
modelo chutou para cima ou para baixo.

In [14]:
comparacao['erro'] = (comparacao['idade real'] - comparacao['idade prevista']).abs()

comparacao.head(5)

,idade real,idade prevista,erro
0,57,39.3,17.7
1,55,50.5,4.5
2,33,38.4,5.4
3,36,39.7,3.7
4,27,42.3,15.3


O **erro médio** é a média dessas distâncias, em todas as pessoas do teste.
O nome técnico é ***mean absolute error*** (erro médio absoluto): "absoluto" porque o sinal é ignorado.

In [15]:
from sklearn.metrics import mean_absolute_error

erro_teste = mean_absolute_error(y_test, y_pred)

print(f'Erro médio no teste: {erro_teste:.1f} anos')

Erro médio no teste: 6.5 anos


### O mesmo erro, usado duas vezes

- Para **escolher** a reta: nos dados de **treino** (é o que o `.fit()` fez).
- Para **julgar** a reta: nos dados de **teste**, que o modelo nunca viu.

In [16]:
erro_treino = mean_absolute_error(y_train, modelo.predict(X_train))

print(f'Erro médio no treino: {erro_treino:.1f} anos')
print(f'Erro médio no teste:  {erro_teste:.1f} anos')

Erro médio no treino: 6.6 anos
Erro médio no teste:  6.5 anos


Aqui deu praticamente igual. **Nem sempre é assim**, e esse é o assunto da aula 4.

---

## 6,5 anos é bom? Bom comparado com o quê?

Para saber se um modelo é bom, é preciso algo para comparar. O **modelo de referência**
(*baseline*) é **o modelo mais simples que alguém poderia usar sem aprender nada com os dados**.

> **Antes de rodar:** se vocês não tivessem modelo nenhum e precisassem chutar **uma única
> idade para todo mundo**, que número chutariam? E quanto esse chute erraria, em média?
>
> **Palpite da turma: ______ anos de erro médio**

In [ ]:
from sklearn.dummy import DummyRegressor

# Qual é o chute mais simples para um número? (lembrem da linha horizontal do slide 46)
# Opções: "mean" (média), "median" (mediana)
referencia = DummyRegressor(strategy="___")
referencia.fit(X_train, y_train)

erro_ref = mean_absolute_error(y_test, referencia.predict(X_test))

print(f'O modelo de referência diz sempre: {referencia.predict(X_test)[0]:.1f} anos')
print(f'Erro médio da referência: {erro_ref:.1f} anos')

O modelo de referência diz sempre: 40.0 anos
Erro médio da referência: 8.4 anos


In [ ]:
pd.DataFrame({
    'modelo': ['Referência: sempre a média', 'Regressão linear'],
    'erro médio (anos)': [round(erro_ref, 1), round(erro_teste, 1)]})

O modelo erra **menos** que a referência: aprender com os dados comprou um pouco menos de
**2 anos** de erro médio.

**Um modelo só vale alguma coisa se bater o seu modelo de referência.** E pode perder: no
slide 46, duas das retas inclinadas erravam mais que a linha horizontal.

Se 2 anos a menos vale a pena ou não, depende do uso. Isso fica para a Parte 3.

---

# Parte 2 — Acertou ou errou? Ou: por quanto errou?

Na aula 1, o Iris foi avaliado contando acertos. Vamos rodar de novo, rapidamente.

In [ ]:
from sklearn import datasets

iris = datasets.load_iris()
X_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
y_iris = iris.target

X_iris_train, X_iris_test, y_iris_train, y_iris_test = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42, stratify=y_iris)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

escala = StandardScaler().fit(X_iris_train)
X_iris_train_s = escala.transform(X_iris_train)
X_iris_test_s = escala.transform(X_iris_test)

knn_iris = KNeighborsClassifier(n_neighbors=3).fit(X_iris_train_s, y_iris_train)
acc_knn_iris = accuracy_score(y_iris_test, knn_iris.predict(X_iris_test_s))

print(f'Acurácia do k-NN no Iris: {acc_knn_iris:.1%}')

### E o modelo de referência para categorias?

Para um número, o chute mais simples é a média. Para uma categoria, é **sempre a categoria mais comum**.

In [ ]:
from sklearn.dummy import DummyClassifier

referencia_iris = DummyClassifier(strategy="most_frequent")
referencia_iris.fit(X_iris_train, y_iris_train)
acc_ref_iris = accuracy_score(y_iris_test, referencia_iris.predict(X_iris_test))

print(f'Acurácia da referência no Iris: {acc_ref_iris:.1%}')
print('Flores de cada espécie no teste:', pd.Series(y_iris_test).value_counts().sort_index().tolist())

Com 10 flores de cada espécie, chutar sempre a mesma acerta 1 em 3. O k-NN chega a 93%:
**este modelo aprendeu muita coisa.**

### E a acurácia do modelo da idade?

In [ ]:
try:
    accuracy_score(y_test, y_pred)
except ValueError as erro:
    print('O scikit-learn recusou:')
    print(erro)

O modelo disse 50,5 anos para uma pessoa de 55. **Isso é um acerto?** Quando a resposta é um
número, não existe "acertou": só existe **a distância**.

| Tarefa | O modelo devolve | Como julgar | Modelo de referência |
|---|---|---|---|
| **Classificação** | uma categoria | acertou ou errou → **acurácia** | sempre a categoria mais comum |
| **Regressão** | um número | a distância → **erro médio** | sempre a média |

**O que o modelo devolve decide como ele é julgado.**

---

# Parte 3 — A pergunta do banco: quem aceita o investimento?

## Etapa 1 — Definição do problema

A coluna `y` diz se a pessoa **aceitou ou não** o investimento. É uma **categoria** →
**classificação**. É a pergunta natural desta base, que ontem ficou de lado.

Ontem, no slide 41: de cada 100 pessoas, **11 disseram sim**. A base é **desbalanceada**.

In [ ]:
y_aceite = df['y']

y_aceite.value_counts(normalize=True).round(3)

## Etapa 3 — Divisão treino / teste

As **mesmas 26 colunas** de antes, agora com outro alvo. Aqui o `stratify` volta: ele mantém
os mesmos 11 em 100 no treino e no teste.

In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cod, y_aceite, test_size=0.2, random_state=42, stratify=y_aceite)

print('Treino:', len(y_train_c), 'pessoas | Teste:', len(y_test_c), 'pessoas')
print('No teste,', (y_test_c == 'yes').sum(), 'pessoas aceitaram')

## O modelo de referência

> **Antes de rodar:** um modelo que diz sempre a resposta mais comum. Quanto ele acerta?
>
> **Palpite da dupla: ______ %**

In [ ]:
# Qual é o chute mais simples para uma categoria?
# Opções: "most_frequent" (a mais comum), "uniform" (sorteio)
referencia_c = DummyClassifier(strategy="___")
referencia_c.fit(X_train_c, y_train_c)
acc_ref_c = accuracy_score(y_test_c, referencia_c.predict(X_test_c))

print('A referência diz sempre:', referencia_c.predict(X_test_c)[0])
print(f'Acurácia da referência: {acc_ref_c:.1%}')

Dizer "não" para todo mundo acerta quase 9 em cada 10. É o slide 15 com números de verdade:
*o modelo aprende que todo cliente deverá ser rejeitado.*

## Um classificador de verdade: a regressão logística

**O nome engana.** Ela faz a mesma soma de pesos da regressão da Parte 1, mas espreme o
resultado entre 0 e 1: vira uma **chance** de a pessoa dizer "sim". Depois, alguém decide a
partir de que chance a resposta vira "sim". **Por padrão, 50%.**

> **Antes de rodar:** quanto ela vai acertar?
>
> **Palpite da dupla: ______ %**

In [ ]:
from sklearn.linear_model import LogisticRegression

logistica = LogisticRegression()
logistica.fit(X_train_c, y_train_c)

y_pred_c = logistica.predict(X_test_c)
acc_log = accuracy_score(y_test_c, y_pred_c)

print(f'Acurácia da regressão logística: {acc_log:.1%}')
print('Pessoas para quem ela disse "sim":', (y_pred_c == 'yes').sum())

**Exatamente a mesma acurácia da referência**, porque ela disse "não" para todo mundo.
Por quê? Vamos olhar as chances que ela calculou.

In [ ]:
# Coluna 1 = chance de 'yes' (a ordem é a de logistica.classes_: 'no', 'yes')
chance = logistica.predict_proba(X_test_c)[:, 1]

print(f'Maior chance calculada para alguém: {chance.max():.0%}')

Ninguém passou de 50%, então ninguém virou "sim". **O modelo virou o modelo de referência.**

In [ ]:
pd.DataFrame({
    'problema': ['Iris: qual espécie?', 'Banco: aceita ou não?'],
    'modelo de referência': [f'{acc_ref_iris:.1%}', f'{acc_ref_c:.1%}'],
    'modelo treinado': [f'{acc_knn_iris:.1%} (k-NN)', f'{acc_log:.1%} (logística)']})

**93% ou 89%: qual é melhor?** Olhando só o número, não dá para saber. O Iris bate a sua
referência por 60 pontos; o banco não bate a sua em nada.

*Um modelo que não funciona bem é um resultado aceitável*, desde que se saiba dizer por quê.
É isto que um resultado honesto parece.

---

# Bom para quê?

O que o banco quer com esse modelo? **Achar as pessoas que aceitariam**, para ligar para elas.

In [ ]:
aceitaram = (y_test_c == 'yes').sum()
achou = ((y_pred_c == 'yes') & (y_test_c == 'yes')).sum()

print('No teste,', aceitaram, 'pessoas aceitaram.')
print('A regressão logística achou', achou, 'delas.')

## Baixando o corte

Por padrão, a resposta vira "sim" quando a chance passa de 50%. **Essa regra é uma escolha,
e dá para mudar.** Vamos ligar para quem tem pelo menos **20%** de chance.

In [ ]:
corte = 0.20
ligar = pd.Series(chance >= corte, index=y_test_c.index)

pd.crosstab(
    y_test_c.map({'no': 'não aceitou', 'yes': 'aceitou'}),
    ligar.map({False: 'modelo disse não', True: 'modelo disse sim'}),
    rownames=[''], colnames=['']).reindex(['não aceitou', 'aceitou'])

Essa tabela tem nome: **matriz de confusão**. Ela mostra os **dois tipos de erro**:

- **não aceitou**, mas o modelo disse sim → uma **ligação perdida**;
- **aceitou**, mas o modelo disse não → uma **venda perdida**.

**Os dois erros não custam a mesma coisa.** Para o banco, uma ligação é barata e um cliente
perdido é caro. **Qual erro é pior não é uma pergunta técnica:** quem decide é quem vai usar o modelo.

## O corte muda tudo

A mesma regressão logística, com cortes diferentes, lado a lado com os dois extremos.

In [ ]:
def resumo(ligar):
    ligar = pd.Series(ligar, index=y_test_c.index)
    aceitou = (y_test_c == 'yes')
    achou = (ligar & aceitou).sum()
    ligacoes = ligar.sum()
    return {
        'ligações': ligacoes,
        'achou (de ' + str(aceitou.sum()) + ')': achou,
        'de cada 100 que aceitariam, achou': round(100 * achou / aceitou.sum()),
        'vendas a cada 100 ligações': round(100 * achou / ligacoes) if ligacoes > 0 else '—',
        'acurácia': f'{(ligar == aceitou).mean():.1%}'}

tabela = pd.DataFrame({
    'Sempre "não" (referência)': resumo([False] * len(y_test_c)),
    'Logística, corte 50% (padrão)': resumo(chance >= 0.50),
    'Logística, corte 30%': resumo(chance >= 0.30),
    'Logística, corte 20%': resumo(chance >= 0.20),
    'Logística, corte 15%': resumo(chance >= 0.15),
    'Logística, corte 10%': resumo(chance >= 0.10),
    'Sempre "sim"': resumo([True] * len(y_test_c)),
}).T

tabela

**Leiam de cima para baixo:**

- Cada passo que acha **mais** gente que aceitaria **baixa** a acurácia. Para este problema, a
  acurácia andava na direção errada o tempo todo.
- **Os dois extremos não servem.** "Sempre não" tem a melhor acurácia e não acha ninguém.
  "Sempre sim" acha todo mundo, mas liga para todo mundo, e aí não precisava de modelo.
  **Onde parar depende de quanto custa uma ligação**, ou de quantas a equipe consegue fazer.
- Ligando ao acaso, **11 em cada 100** ligações viram venda. Com o corte em 30%, são **33 em
  cada 100**. **O modelo aprendeu alguma coisa**, e a acurácia escondia isso.

A coluna *de cada 100 que aceitariam, quantos o modelo achou* tem nome: ***recall***.
A coluna *vendas a cada 100 ligações* também tem nome, e fica para a monitoria de quinta.

### Experimentem: o corte de vocês

A equipe do banco consegue fazer **umas 1.300 ligações**. Que corte vocês escolheriam?
Rodem com o corte de vocês e vejam o que muda.

In [ ]:
corte_de_voces = ___    # um número entre 0 e 1. Ex.: 0.25 quer dizer 25%

pd.DataFrame({f'Logística, corte {corte_de_voces:.0%}': resumo(chance >= corte_de_voces)}).T

**Onde cortar é uma decisão**, e ela vai na **seção 4 do relatório** (*que decisão a saída
alimenta? qual ponto de corte vocês escolheriam?*).

Para quem está fazendo regressão (plano de saúde, vinho como número), é a mesma ideia: a partir
de que valor previsto vocês agiriam?

---

# E quando ninguém escreveu a resposta?

Tudo o que foi feito hoje dependeu de uma coisa: **existir uma resposta certa para comparar**.

| | Supervisionado | Não supervisionado |
|---|---|---|
| **A resposta** | alguém escreveu (a espécie, a idade, o `y`) | ninguém escreveu |
| **Como julgar** | comparando com a resposta: acertou, ou por quanto errou | não há o que comparar: você julga se o resultado é útil |
| **O que define a pergunta** | a coluna-alvo | **as colunas que você escolhe** para dizer o que é "parecido" |

No não supervisionado, o modelo devolve quais linhas são parecidas entre si e um retrato de cada
grupo. **Dar nome ao grupo é trabalho seu.** O algoritmo mais usado para isso se chama **k-means**,
e é assunto da aula 6.

---

# Fechando

Duas perguntas para fazer a qualquer modelo, a começar pelo de vocês:

> **Bom comparado com o quê?** — o modelo de referência.
>
> **Bom para quê?** — a medida vem do que o problema precisa.

### Para o projeto de vocês

1. Qual é o **modelo de referência** de vocês? (a categoria mais comum, ou a média)
2. O modelo de vocês **acerta ou erra**, ou **erra por uma distância**? Então, que medida usar?
3. No problema de vocês, **qual erro é pior**? Onde vocês cortariam?

**Sugestão, se o grupo quiser adiantar:** rodem o modelo de referência na base de vocês
(`DummyClassifier` ou `DummyRegressor`, é copiar deste notebook) e comparem com o primeiro modelo
que vocês tiverem. Isso já é metade da seção 3.